# 01 — HAR-RV (weekly silver volatility)

First two models on the volatility ladder:

- **Naïve** — $\hat{\text{RV}}_t = \text{RV}_{t-1}$. The floor. Volatility has a strong
  AR(1) component, so last week's RV is already a hard baseline to beat.
- **HAR-RV** (Corsi 2009) — OLS regression of RV on its short / medium / long trailing
  averages.

Features come from `volatility_weekly.csv`, built in `00_features.ipynb` — run that
notebook first.


## Setup


In [1]:
import sys, os
sys.path.append(os.path.abspath('../../src'))

import numpy as np
import pandas as pd
import statsmodels.api as sm
from vol_utils import vol_evaluate, vol_period_metrics
from eval_utils import PERIODS
import warnings; warnings.filterwarnings('ignore')

frame = pd.read_csv('../../data/processed/volatility_weekly.csv',
                    parse_dates=['Date']).set_index('Date')

train_df = frame[frame['split'] == 'train']
val_df   = frame[frame['split'] == 'val']
test_df  = frame[frame['split'] == 'test']
trval_df = frame[frame['split'] != 'test']     # HAR-RV is plain OLS -> train+val fitted as one

FEATS_HAR = ['rv_w_lag1', 'rv_m_lag1', 'rv_q_lag1']
print(f'train+val: {len(trval_df)}   test: {len(test_df)}')


train+val: 405   test: 175


## 1. Naïve baseline — $\hat{\text{RV}}_t = \text{RV}_{t-1}$

The naïve prediction is simply last week's RV, which is exactly the `rv_w_lag1` column.
Because volatility is highly persistent this is a genuinely strong baseline — every
other model has to beat it to justify itself.

(Its DCA is ≈ 0 by construction: predicting last week's RV implies *no* change, so the
naïve model can never call a direction. It is a floor for RMSE/MAE, not for DCA.)


In [2]:
y_test    = test_df['target'].values
prev_test = test_df['rv_w_lag1'].values        # RV_{t-1}

results = []
results.append(vol_evaluate('Naive (RV_t-1)', y_test, prev_test, prev_test))


Naive (RV_t-1)                  RMSE=0.03979  MAE=0.02013  R2=-0.095  DCA=0.000


## 2. HAR-RV (Corsi 2009)

**HAR-RV** stands for the *Heterogeneous Autoregressive model of Realised Volatility*.
It is a simple linear regression that predicts realised volatility from its own past
values measured over several horizons. The "heterogeneous" in the name refers to the
economic story behind it: different market participants operate on different time
horizons — short-term traders care about last week, medium-term traders about last
month, longer-term traders about last quarter — so the volatility observed today
reflects a mixture of all of those horizons.

Concretely it is OLS on the three HAR features built in `00_features.ipynb`:

$$\text{RV}_t = \beta_0 + \beta_w \text{RV}^{(w)}_{t-1} + \beta_m \text{RV}^{(m)}_{t-1} + \beta_q \text{RV}^{(q)}_{t-1} + \varepsilon_t$$

Despite its simplicity HAR-RV is the standard volatility benchmark in the literature,
because those three lags capture the long-memory persistence of volatility well. Train
and val are pooled for the final fit since OLS has no hyperparameters to tune.


In [3]:
X_tr = sm.add_constant(trval_df[FEATS_HAR])
y_tr = trval_df['target']
X_te = sm.add_constant(test_df[FEATS_HAR])

har_fit = sm.OLS(y_tr, X_tr).fit()
print(har_fit.summary().tables[1])

har_pred = har_fit.predict(X_te).values
results.append(vol_evaluate('HAR-RV', y_test, har_pred, prev_test))


                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0085      0.003      3.196      0.002       0.003       0.014
rv_w_lag1      0.0668      0.062      1.074      0.284      -0.056       0.189
rv_m_lag1      0.3971      0.126      3.155      0.002       0.150       0.644
rv_q_lag1      0.2799      0.129      2.173      0.030       0.027       0.533
HAR-RV                          RMSE=0.03153  MAE=0.01576  R2=+0.313  DCA=0.714


## 3. Sub-period breakdown

RMSE and DCA for HAR-RV split by calendar year, using the shared `PERIODS` definition
from `eval_utils`. This shows whether the model holds up across the choppy 2023, the
2024 bull start, the 2025 bull run, and 2026 YTD.


In [4]:
period_har = vol_period_metrics(y_test, har_pred, prev_test, test_df.index, PERIODS)
period_har.to_csv('../../data/processed/period_har_volatility.csv')
period_har.round(4)


,n,RMSE,MAE,DCA
Period,,,,
2023 (choppy),52,0.0148,0.0122,0.6923
2024 (bull start),52,0.0145,0.0114,0.7885
2025 (bull run),52,0.0222,0.0155,0.6923
2026 (YTD),19,0.0815,0.0381,0.6316
── Full test ──,175,0.0315,0.0158,0.7143


## 4. Save outputs

- `metrics_har_volatility.csv` — Naïve + HAR-RV headline metrics
- `pred_har_volatility.csv` — test-set predictions, consumed by `evaluation.ipynb`


In [5]:
pd.DataFrame(results).to_csv('../../data/processed/metrics_har_volatility.csv', index=False)

pred_har = pd.DataFrame({'actual': y_test, 'prev': prev_test,
                         'naive': prev_test, 'har': har_pred}, index=test_df.index)
pred_har.to_csv('../../data/processed/pred_har_volatility.csv', index_label='Date')
print('Saved metrics_har_volatility.csv + pred_har_volatility.csv')
pd.DataFrame(results).round(5)


Saved metrics_har_volatility.csv + pred_har_volatility.csv


,model,rmse,mae,r2,dca
0,Naive (RV_t-1),0.03979,0.02013,-0.09462,0.00000
1,HAR-RV,0.03153,0.01576,0.31255,0.71429
